# Molecular representation comparison

Predict an RDKit-computed property (logP) from three representations of the same molecules: Morgan fingerprints, tokenized SMILES with a 1D CNN, and tokenized SMILES with an LSTM. This runs fully offline on a built-in SMILES set; `scripts/benchmark.py` runs the same comparison on a larger downloaded set.

In [1]:
from molprop import build_dataset, random_split, Trainer, set_seed
from molprop.models import FingerprintMLP, SmilesCNN, SmilesRNN

set_seed(0)
dataset = build_dataset(target='logp', n_bits=1024)
train_set, val_set, test_set = random_split(dataset, seed=0)
vocab = len(dataset.tokenizer)
len(train_set), len(val_set), len(test_set), vocab

(79, 9, 11, 24)

In [2]:
factories = {
    'FingerprintMLP': lambda: FingerprintMLP(n_bits=1024),
    'SmilesCNN': lambda: SmilesCNN(vocab),
    'SmilesRNN': lambda: SmilesRNN(vocab),
}
for name, make in factories.items():
    set_seed(0)
    trainer = Trainer(make(), lr=1e-3, weight_decay=1e-5)
    trainer.fit(train_set, val_set, epochs=60, batch_size=16, verbose=False)
    print(name, trainer.evaluate(test_set))

FingerprintMLP {'rmse': 0.6229474558098251, 'mae': 0.45228714834560046, 'r2': 0.5406099654026888}


SmilesCNN {'rmse': 0.5911773684125894, 'mae': 0.3260634487325495, 'r2': 0.5862725495391805}


SmilesRNN {'rmse': 0.4676418712191957, 'mae': 0.3577387983148748, 'r2': 0.7411158175671634}
